# 7 - Evaluate a policy

Notebooks 1 to 6 record data, train a policy, deploy it and orchestrate a fleet.
None of them answers the question that decides whether any of it worked: **is this policy any good?**

This notebook is that answer. It scores a policy on a task, reads the result per attempt rather
than as an average, and defines a new task as data. No hardware, no GPU, no Hugging Face
credentials.

Evaluation here has three layers, and this notebook is **layer one**:

| Layer | What it decides | Where it lives |
|---|---|---|
| **1. Deterministic conditions** | Did the task succeed or fail, in how many steps, at what reward. Authoritative. | This notebook |
| **2. Judge annotation** | Was it done *well* - smooth or jerky, clean or lucky. Cannot overrule layer 1. | [`examples/17_judge_recorded_episodes.py`](../17_judge_recorded_episodes.py) |
| **3. Human agreement** | How much to trust layer 2. | Same example |

**Install:** `uv pip install -U "strands-robots[sim-mujoco]"`

Part of the [examples/notebooks](./README.md) getting-started series.


In [ ]:
import os
import sys

# macOS uses the 'cgl' offscreen GL backend; Linux headless uses 'egl'.
os.environ.setdefault("MUJOCO_GL", "cgl" if sys.platform == "darwin" else "egl")


## 1. What tasks are there to score?

A task is registered under a name. `register_builtin_benchmarks()` adds the five shipped
locomotion tasks, and `list_benchmarks()` reports what is registered plus which robot each one
targets - enough to pick one without instantiating anything.

`max_steps` is the per-attempt horizon. An attempt that has not succeeded by then is over.


In [ ]:
from strands_robots.simulation import register_builtin_benchmarks
from strands_robots.simulation.benchmark import list_benchmarks

register_builtin_benchmarks()  # idempotent

registry = list_benchmarks()
print(f"{'task':22} {'robot':16} max_steps")
for name, meta in sorted(registry.items()):
    print(f"{name:22} {meta['default_robot']:16} {meta['max_steps']}")


## 2. Score a policy

`evaluate_benchmark` builds the task's scene, runs the policy for up to `max_steps` per attempt,
checks the success and failure conditions every step, and accumulates the task's reward terms.

We score the `mock` policy first, deliberately. Mock emits sinusoids and does not walk, so its
score is the **floor**: the number a policy that does nothing useful gets on this task. Without a
floor you cannot tell a working policy from a task that is trivially satisfiable.


In [ ]:
from strands_robots import Robot

TASK = "go2_walk_forward"
robot_name = registry[TASK]["default_robot"]

sim = Robot(robot_name, mesh=False)
result = sim.evaluate_benchmark(TASK, policy_provider="mock", n_episodes=3, seed=0)

if result.get("status") == "error":
    raise RuntimeError("; ".join(c.get("text", "") for c in result.get("content", [])))

metrics = next(c["json"] for c in result["content"] if "json" in c)
print(f"{TASK} on {robot_name}, policy='mock' (the floor)")
print(f"  episodes_completed : {metrics['episodes_completed']}")
print(f"  success_rate       : {metrics['success_rate']}")
print(f"  avg_reward         : {metrics['avg_reward']}")
print(f"  avg_steps          : {metrics['avg_steps']}")


## 3. Read the attempts, not the average

`success_rate` tells you something is wrong. It does not tell you **what**. The per-attempt rows
do, and they are in the same result under `episodes`.

Each row carries `seed`, which is the useful part: every attempt is seeded from the master `seed`
you passed, so any single row can be replayed on its own without re-running the set.

`success` and `failure` are separate fields, not opposites. An attempt that ran out of steps is
`success=False, failure=False` - it neither achieved the goal nor triggered a failure condition.
That distinction is what tells a timeout apart from a fall.


In [ ]:
episodes = metrics["episodes"]
print(f"{'ep':>3} {'success':>8} {'failure':>8} {'steps':>7} {'reward':>10} {'seed':>6}  outcome")
for row in episodes:
    if row["success"]:
        outcome = "achieved the goal"
    elif row["failure"]:
        outcome = "failure condition fired"
    else:
        outcome = "ran out of steps"
    print(
        f"{row['episode']:>3} {str(row['success']):>8} {str(row['failure']):>8} "
        f"{row['steps']:>7} {row['cumulative_reward']:>10} {row['seed']:>6}  {outcome}"
    )


## 4. Define your own task, as data

A task is a dict, not a Python subclass. `success`, `failure` and `dense_reward` are composed
from named entries in a **closed registry** of conditions and reward terms
(`strands_robots.simulation.predicates`). Nothing in a spec is ever executed as code, which is
what makes a spec safe to load from a JSON or YAML file an agent produced.

The task below is a harder variant of `go2_walk_forward`: walk past 4 m at a 1.5 m/s target, and
fail on a tip or a collapsed base. Same authoring path the shipped tasks use.


In [ ]:
from strands_robots.simulation.benchmark import register_benchmark
from strands_robots.simulation.benchmark_spec import DeclarativeBenchmark

SPEC = {
    "name": "go2_walk_far_fast",
    "instruction": "Walk forward at 1.5 m/s and cover at least 4 meters without tipping.",
    "default_robot": "unitree_go2",
    "supported_robots": ["unitree_go2"],
    "max_steps": 400,
    "success": {"all": [{"predicate": "base_beyond_x", "x": 4.0}]},
    "failure": {
        "any": [
            {"predicate": "base_tipped", "tol": 0.7},
            {"predicate": "base_below_z", "z": 0.18},
        ]
    },
    "dense_reward": [
        {"predicate": "base_velocity_tracking", "vx": 1.5, "lin_weight": 1.0, "ang_weight": 0.5},
        {"predicate": "base_height", "target": 0.32, "weight": 0.5},
        {"predicate": "base_orientation", "weight": 0.5},
    ],
}

benchmark = DeclarativeBenchmark.from_dict(SPEC)  # compiles now, not mid-episode
register_benchmark(benchmark.name, benchmark)
print(f"registered {benchmark.name!r} for {benchmark.default_robot}")

custom = sim.evaluate_benchmark(benchmark.name, policy_provider="mock", n_episodes=2, seed=0)
custom_metrics = next(c["json"] for c in custom["content"] if "json" in c)
print(f"  success_rate : {custom_metrics['success_rate']}  (mock does not walk, so 0 is expected)")
print(f"  avg_reward   : {custom_metrics['avg_reward']}")


## 5. A broken spec is refused before anything runs

Both failures below raise from `from_dict`, before a scene is built or a step is taken. That
matters more than the tidy error message: a spec that compiled and then silently evaluated to
`False` forever would report a perfectly plausible 0% success rate, and you would go looking at
the policy.

The first message lists every valid condition name, so a typo is self-correcting.


In [ ]:
for label, broken in [
    ("unknown condition name", {**SPEC, "success": {"all": [{"predicate": "base_beyond_z", "z": 4.0}]}}),
    ("wrong argument name", {**SPEC, "success": {"all": [{"predicate": "base_beyond_x", "distance": 4.0}]}}),
]:
    try:
        DeclarativeBenchmark.from_dict({**broken, "name": "should_not_register"})
    except ValueError as exc:
        print(f"{label}: refused at compile time")
        print(f"  {str(exc)[:160]}")
    else:
        raise AssertionError(f"{label} was accepted; the closed registry is not being enforced")


## Where layers 2 and 3 live

Everything above is layer one: conditions over simulator state, and it is authoritative. What it
cannot tell you is whether a run was *good*. Two attempts can both succeed with one clean and one
lucky, and if you train on the lucky one you make the next policy worse.

That is what layers two and three are for, and they ship in
[`examples/17_judge_recorded_episodes.py`](../17_judge_recorded_episodes.py):

- A **judge agent** reads the recorded episode and adds a quality grade (`low` / `medium` /
  `high`) and a failure tag from a fixed taxonomy. It writes into its own block of a label
  sidecar, so it is structurally unable to change the verdict from layer one; a disagreement is
  recorded as a dispute for a human.
- **`measure_agreement`** compares the judge against human grades on a holdout, so the judge's
  numbers are reported with a known level of trust rather than assumed.
- **`filter_episodes`** then selects the successful, high-quality subset to retrain on, which is
  what closes the loop from evaluation back into the next model.

You only need layers two and three when you are letting an automated grader decide what a policy
trains on. For "does this work", layer one is the whole answer.


## Next

- **[`examples/10_evaluate_benchmark.py`](../10_evaluate_benchmark.py)** - the same scoring loop
  as a script.
- **[`examples/11_author_a_benchmark.py`](../11_author_a_benchmark.py)** - the authoring path on
  its own.
- **[`examples/17_judge_recorded_episodes.py`](../17_judge_recorded_episodes.py)** - layers two
  and three end to end: record, verdict, judge, agreement, filter, retrain.
- Swap `policy_provider="mock"` for `"groot"`, `"cosmos3"` or `"lerobot_local"` with a checkpoint
  in `policy_config` to score a trained policy on the identical task. Those need a GPU; the
  scoring code is unchanged.
